# Diplomado de Grado en Análisis de Datos Aplicado a la Toma de Decisiones
## UNICOLOMBO — Fundación Universitaria Colombo Internacional · Cartagena
**Módulo 2:** Preparación y Análisis Exploratorio de Datos  
**Proyecto Integrador Final:** Pipeline Completo DataRetail LATAM  
**Docente:** Ing. Heyder Medrano Olier | **Período:** 2026 — Vacaciones

---
**Entrega:** 8 de julio de 2026 · 23:59 (hora Colombia)  
**Valor:** 5.0 puntos  

**Estudiante(s):** Anisley Povea García, Jefferson Arrieta Durán  

> Este proyecto integra los aprendizajes de las 4 sesiones del Módulo 2:
> S01 (Diagnóstico) → S02 (Limpieza) → S03 (Transformación) → S04 (EDA)
> en un **pipeline automático y reproducible** sobre el dataset DataRetail LATAM.

## Objetivo del Proyecto

Construir un **pipeline de datos end-to-end** que:
1. Ingeste el dataset DataRetail LATAM bruto
2. Diagnostique su calidad (S01)
3. Lo limpie automáticamente (S02)
4. Lo transforme para análisis (S03)
5. Genere insights de negocio (S04)
6. Produzca un reporte ejecutivo para el CTO

**Criterios de evaluación:**
- Pipeline funcional y reproducible (Kernel → Restart & Run All)
- Mínimo 5 insights de negocio originales
- Visualizaciones claras y profesionales
- Código limpio y documentado
- Conclusiones en lenguaje ejecutivo

## Parte 1 — Librerías e Importaciones

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns, scipy.stats as stats
import missingno as msno, warnings, time
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from datetime import datetime, timedelta
import random
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format','{:.2f}'.format)
print('Librerías importadas correctamente')
print(f'pandas: {pd.__version__} | numpy: {np.__version__}')

## Parte 2 — Clase DataPipeline

Encapsulamos todo el proceso en una clase Python que ejecuta el pipeline de forma modular y trazable. Cada paso registra métricas en un log.

In [ ]:
from datetime import datetime

class DataPipeline:
    """
    Pipeline de procesamiento para el dataset DataRetail LATAM.

    Permite realizar las etapas de diagnóstico, limpieza,
    transformación y análisis exploratorio de datos (EDA),
    registrando cada proceso ejecutado.
    """

    def __init__(self, nombre='DataRetail_LATAM'):
        """
        Inicializa el pipeline.

        Parameters
        ----------
        nombre : str
            Nombre del proyecto o dataset.
        """
        self.nombre = nombre
        self.df_raw = None
        self.df_clean = None
        self.df_trans = None
        self.metricas = {}
        self.log = []
        print(f'Pipeline [{nombre}] inicializado')

    def _registrar(self, paso, detalle):
        """
        Registra en el log cada actividad realizada
        durante la ejecución del pipeline.

        Parameters
        ----------
        paso : str
            Nombre de la etapa ejecutada.
        detalle : str
            Descripción de la acción realizada.
        """
        ts = datetime.now().strftime('%H:%M:%S')
        self.log.append(f'[{ts}] {paso}: {detalle}')
        print(f'[{ts}] {paso}: {detalle}')

    def imprimir_log(self):
        """
        Muestra el historial de ejecución del pipeline.
        """
        print('\n=== LOG DEL PIPELINE ===')
        for entrada in self.log:
            print(f'  {entrada}')
        print('========================')

## Parte 3 — Paso 1: Ingesta de Datos

In [ ]:
def ingestar(self):
    """
    Genera un dataset sintético de DataRetail LATAM para el proyecto.

    Crea registros de ventas con información de clientes, productos,
    ciudades, canales de venta y variables comerciales. Además,
    introduce intencionalmente problemas de calidad como valores
    nulos, duplicados y outliers para simular un escenario real.
    """

    # Comentario de negocio:
    # Esta etapa simula la recepción de información comercial desde
    # diferentes sucursales y canales de venta. Se incluyen errores
    # de calidad para representar situaciones comunes en bases de
    # datos empresariales antes de su procesamiento.

    import random
    from datetime import datetime, timedelta

    np.random.seed(42)
    random.seed(42)

    N = 2000

    ciudades = [
        'Bogotá','Medellín','Cali','Barranquilla','Cartagena',
        'Lima','Ciudad de México','Buenos Aires','Santiago','Montevideo'
    ]

    ciu_s = ciudades + [
        'bogota','BOGOTÁ','Medellin','cali ','SANTIAGO'
    ]

    paises = [
        'Colombia','Perú','México','Argentina','Chile','Uruguay'
    ]

    cats = [
        'Computación','Periféricos','Audio',
        'Telefonía','Accesorios','Gaming','Componentes'
    ]

    canales = [
        'Tienda Física','E-commerce',
        'Distribuidor','Corporativo','Marketplace'
    ]

    can_s = canales + [
        'tienda fisica','E-Comerce','CORPORATIVO'
    ]

    prods = [
        'Laptop Pro','Tablet Ultra','Monitor 4K',
        'Teclado Inalámbrico','Auriculares BT',
        'Parlante Portátil','Smartphone Galaxy',
        'Cámara Web HD','Smartwatch Pro',
        'Mouse Gaming','SSD 1TB','GPU RTX 4060'
    ]

    precios = {
        'Laptop Pro':1200,
        'Tablet Ultra':450,
        'Monitor 4K':380,
        'Teclado Inalámbrico':75,
        'Auriculares BT':120,
        'Parlante Portátil':90,
        'Smartphone Galaxy':680,
        'Cámara Web HD':95,
        'Smartwatch Pro':250,
        'Mouse Gaming':65,
        'SSD 1TB':130,
        'GPU RTX 4060':580
    }

    rows = []

    for i in range(N):

        prod = random.choice(prods)
        cat = cats[prods.index(prod) % len(cats)]

        pais = random.choice(paises)

        ciudad = random.choice(ciu_s) if random.random() < 0.15 else random.choice(ciudades)
        canal = random.choice(can_s) if random.random() < 0.10 else random.choice(canales)

        qty = random.randint(1,20)

        precio = round(precios[prod] * np.random.uniform(0.8,1.3),2)

        desc = round(
            random.choice([0,0,0,0.05,0.10,0.15,0.20,0.30,0.35,0.50]),
            2
        )

        total = round(qty * precio * (1-desc),2)

        margen = round(
            total * np.random.uniform(-0.05,0.35),
            2
        )

        fecha = datetime(2024,1,1) + timedelta(days=random.randint(0,729))

        rows.append({
            'id_venta':f'V{i+1:05d}',
            'fecha_venta':fecha,
            'id_cliente':f'C{random.randint(1,500):04d}',
            'nombre_cliente':f'Cliente {random.randint(1,500)}',
            'ciudad':ciudad,
            'pais':pais,
            'id_producto':f'P{prods.index(prod)+1:02d}',
            'nombre_producto':prod,
            'categoria':cat,
            'canal_venta':canal,
            'cantidad':qty,
            'precio_unitario':precio,
            'descuento':desc,
            'total_venta':total,
            'margen_utilidad':margen
        })

    df = pd.DataFrame(rows)

    idx_p = np.random.choice(df.index,size=int(N*0.06),replace=False)
    idx_t = np.random.choice(df.index,size=int(N*0.04),replace=False)
    idx_m = np.random.choice(df.index,size=int(N*0.03),replace=False)

    df.loc[idx_p,'precio_unitario'] = np.nan
    df.loc[idx_t,'total_venta'] = np.nan
    df.loc[idx_m,'margen_utilidad'] = np.nan

    dup_idx = np.random.choice(df.index,size=40,replace=False)
    df = pd.concat([df,df.loc[dup_idx].copy()],ignore_index=True)

    out_idx = np.random.choice(df.index,size=10,replace=False)
    df.loc[out_idx,'total_venta'] = np.random.uniform(50000,200000,10)

    self.df_raw = df.sample(frac=1,random_state=42).reset_index(drop=True)

    self.metricas['n_raw'] = len(self.df_raw)

    self._registrar(
        'INGESTA',
        f'{len(self.df_raw):,} filas x {self.df_raw.shape[1]} columnas'
    )

    return self

DataPipeline.ingestar = ingestar

print("Método ingestar() registrado")

## Parte 4 — Paso 2: Diagnóstico (S01)

In [ ]:
def diagnosticar(self):
    """
    Realiza el diagnóstico inicial de la calidad de los datos.

    Evalúa la presencia de valores nulos, registros duplicados,
    valores atípicos (outliers) y descuentos superiores al límite
    establecido. Además, calcula un score de calidad de los datos.
    """

    # Comentario de negocio:
    # Un diagnóstico temprano permite identificar problemas de calidad
    # que podrían afectar los análisis y la toma de decisiones de la empresa.

    df = self.df_raw.copy()

    df['fecha_venta'] = pd.to_datetime(df['fecha_venta'], errors='coerce')
    df['precio_unitario'] = pd.to_numeric(df['precio_unitario'], errors='coerce')

    # Valores nulos
    nulos = df.isna().sum().sum()

    # Registros duplicados
    dup = df.duplicated().sum()

    # Detección de outliers en total_venta
    s = pd.to_numeric(df['total_venta'], errors='coerce').dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1

    out = ((s < Q1 - 1.5 * IQR) | (s > Q3 + 1.5 * IQR)).sum()

    # Descuentos mayores al 30 %
    desc_viol = (df['descuento'] > 0.30).sum()

    # Score de calidad
    pct_n = df.isna().mean().mean() * 100

    score_inicial = (
        100
        - min(pct_n * 3, 30)
        - min(dup / len(df) * 5 * 100, 20)
        - min(out / len(df) * 1.5 * 100, 15)
    )

    self.metricas.update({
        'nulos': nulos,
        'duplicados': dup,
        'outliers': out,
        'desc_viol': desc_viol,
        'score_inicial': round(score_inicial, 1)
    })

    self._registrar(
        'DIAGNÓSTICO',
        f'Score: {score_inicial:.1f}/100 | Nulos: {nulos} | Dup: {dup} | Out: {out}'
    )

    return self


DataPipeline.diagnosticar = diagnosticar

print("Método diagnosticar() registrado")

## Parte 5 — Paso 3: Limpieza (S02)

In [ ]:
def limpiar(self):
    """
    Realiza la limpieza del dataset.

    Elimina registros duplicados, corrige tipos de datos,
    imputa valores faltantes, trata valores atípicos,
    estandariza variables categóricas y recalcula el
    valor total de las ventas.
    """

    # Comentario de negocio:
    # La limpieza de datos garantiza información consistente y confiable,
    # permitiendo generar análisis e indicadores precisos para la toma
    # de decisiones empresariales.

    df = self.df_raw.copy()

    # Conversión de tipos de datos
    df['fecha_venta'] = pd.to_datetime(df['fecha_venta'], errors='coerce')
    df['precio_unitario'] = pd.to_numeric(df['precio_unitario'], errors='coerce')
    df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').astype('Int64')

    # Eliminar registros duplicados
    n_antes = len(df)
    df = df.drop_duplicates(keep='first').reset_index(drop=True)

    # Imputación del precio por la mediana de la categoría
    med_cat = df.groupby('categoria')['precio_unitario'].transform('median')
    df['precio_unitario'] = (
        df['precio_unitario']
        .fillna(med_cat)
        .fillna(df['precio_unitario'].median())
    )

    # Recalcular ventas faltantes
    mask = df['total_venta'].isna()
    df.loc[mask, 'total_venta'] = (
        df.loc[mask, 'cantidad']
        * df.loc[mask, 'precio_unitario']
        * (1 - df.loc[mask, 'descuento'])
    ).round(2)

    # Imputación del margen de utilidad
    med_m = df.groupby('categoria')['margen_utilidad'].transform('median')
    df['margen_utilidad'] = (
        df['margen_utilidad']
        .fillna(med_m)
        .fillna(df['margen_utilidad'].median())
    )

    # Tratamiento de outliers mediante el método IQR
    for col in ['total_venta', 'precio_unitario']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        df[col] = df[col].clip(
            lower=Q1 - 1.5 * IQR,
            upper=Q3 + 1.5 * IQR
        )

    # Estandarizar nombres de ciudades
    df['ciudad'] = (
        df['ciudad']
        .str.strip()
        .str.title()
        .replace({
            'Bogota': 'Bogotá',
            'Medellin': 'Medellín',
            'Ciudad De Mexico': 'Ciudad de México'
        })
    )

    # Estandarizar canales de venta
    df['canal_venta'] = (
        df['canal_venta']
        .str.strip()
        .replace({
            'tienda fisica': 'Tienda Física',
            'E-Comerce': 'E-commerce',
            'CORPORATIVO': 'Corporativo'
        })
    )

    # Limitar descuentos al 30 %
    df['descuento'] = df['descuento'].clip(upper=0.30)

    # Recalcular el total de venta
    df['total_venta'] = (
        df['cantidad']
        * df['precio_unitario']
        * (1 - df['descuento'])
    ).round(2)

    self.df_clean = df

    self.metricas['n_clean'] = len(df)
    self.metricas['score_post'] = round(
        100 - df.isna().mean().mean() * 300,
        1
    )

    self._registrar(
        'LIMPIEZA',
        f'{n_antes - len(df)} filas eliminadas | '
        f'Nulos: {df.isna().sum().sum()} | '
        f'Score: {self.metricas["score_post"]}/100'
    )

    return self


DataPipeline.limpiar = limpiar

print("Método limpiar() registrado")

## Parte 6 — Paso 4: Transformación (S03)

In [ ]:
def transformar(self):
    """
    Realiza la transformación del dataset.

    Crea nuevas variables de negocio, variables temporales,
    realiza codificación de variables categóricas y aplica
    escalado para facilitar el análisis y futuros modelos
    de Machine Learning.
    """

    # Comentario de negocio:
    # La transformación crea variables que permiten comprender
    # mejor el comportamiento comercial y preparar los datos
    # para análisis avanzados y modelos predictivos.

    df = self.df_clean.copy()

    # ==========================
    # Variables temporales
    # ==========================
    df['anio'] = df['fecha_venta'].dt.year
    df['mes'] = df['fecha_venta'].dt.month
    df['trimestre'] = df['fecha_venta'].dt.quarter
    df['dia_semana'] = df['fecha_venta'].dt.dayofweek
    df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)

    # ==========================
    # Variables de negocio
    # ==========================
    df['margen_pct'] = (
        df['margen_utilidad']
        / df['total_venta'].replace(0, np.nan)
    ).fillna(0).round(4)

    df['precio_neto'] = (
        df['precio_unitario']
        * (1 - df['descuento'])
    ).round(2)

    df['total_x_cliente'] = (
        df.groupby('id_cliente')['total_venta']
        .transform('sum')
        .round(2)
    )

    df['frecuencia_cliente'] = (
        df.groupby('id_cliente')['id_venta']
        .transform('count')
    )

    # ==========================
    # Segmentación de precios
    # ==========================
    df['segmento_precio'] = pd.cut(
        df['precio_unitario'],
        bins=[0,100,400,2000],
        labels=['Económico','Medio','Premium']
    )

    # ==========================
    # Encoding
    # ==========================
    from sklearn.preprocessing import LabelEncoder, RobustScaler

    le = LabelEncoder()

    df['categoria_num'] = le.fit_transform(df['categoria'].astype(str))

    # ==========================
    # Escalado (RobustScaler)
    # ==========================
    scaler = RobustScaler()

    columnas_escalar = [
        'precio_unitario',
        'cantidad',
        'total_venta',
        'margen_utilidad'
    ]

    df[[c + '_scaled' for c in columnas_escalar]] = scaler.fit_transform(
        df[columnas_escalar]
    )

    self.df_trans = df

    nuevas = [
        'anio',
        'mes',
        'trimestre',
        'dia_semana',
        'es_fin_semana',
        'margen_pct',
        'precio_neto',
        'total_x_cliente',
        'frecuencia_cliente',
        'segmento_precio',
        'categoria_num',
        'precio_unitario_scaled',
        'cantidad_scaled',
        'total_venta_scaled',
        'margen_utilidad_scaled'
    ]

    self._registrar(
        'TRANSFORMACIÓN',
        f'{len(nuevas)} variables nuevas | Total columnas: {df.shape[1]}'
    )

    return self


DataPipeline.transformar = transformar

print("Método transformar() registrado")

## Parte 7 — Paso 5: EDA y Generación de Insights (S04)

In [ ]:
def analizar(self):
    """
    Realiza el Análisis Exploratorio de Datos (EDA).

    Calcula la segmentación RFM de los clientes e identifica
    los principales insights del negocio, como la categoría
    más vendida, el canal más rentable, el mejor mes de ventas
    y el margen promedio.
    """

    # Comentario de negocio:
    # El EDA permite descubrir patrones de comportamiento de los
    # clientes y del negocio para apoyar la toma de decisiones
    # estratégicas y comerciales.

    df = self.df_trans.copy()

    # ==========================
    # Análisis RFM
    # ==========================
    fecha_referencia = df['fecha_venta'].max()

    rfm = (
        df.groupby('id_cliente')
        .agg(
            recencia=('fecha_venta', lambda x: (fecha_referencia - x.max()).days),
            frecuencia=('id_venta', 'count'),
            monetario=('total_venta', 'sum')
        )
        .reset_index()
    )

    # ==========================
    # Puntajes RFM
    # ==========================
    rfm['R_score'] = pd.qcut(
        rfm['recencia'],
        q=4,
        labels=[4, 3, 2, 1]
    ).astype(int)

    rfm['F_score'] = pd.qcut(
        rfm['frecuencia'],
        q=4,
        labels=[1, 2, 3, 4]
    ).astype(int)

    rfm['M_score'] = pd.qcut(
        rfm['monetario'],
        q=4,
        labels=[1, 2, 3, 4]
    ).astype(int)

    rfm['RFM'] = (
        rfm['R_score']
        + rfm['F_score']
        + rfm['M_score']
    )

    # ==========================
    # Segmentación de clientes
    # ==========================
    def clasificar(score):
        if score >= 10:
            return 'Champions'
        elif score >= 7:
            return 'Loyal'
        elif score >= 5:
            return 'Potential'
        else:
            return 'At Risk'

    rfm['segmento'] = rfm['RFM'].apply(clasificar)

    self.rfm = rfm

    # ==========================
    # Insights de negocio
    # ==========================
    mejor_cat = (
        df.groupby('categoria')['total_venta']
        .sum()
        .idxmax()
    )

    mejor_canal = (
        df.groupby('canal_venta')['total_venta']
        .sum()
        .idxmax()
    )

    mejor_mes = (
        df.groupby('mes')['total_venta']
        .sum()
        .idxmax()
    )

    champions = (
        rfm['segmento']
        .eq('Champions')
        .sum()
    )

    margen_promedio = df['margen_pct'].mean() * 100

    self.insights = [
        f'Categoría líder: {mejor_cat}',
        f'Canal más rentable: {mejor_canal}',
        f'Mes con mayores ventas: {mejor_mes}',
        f'Clientes Champions identificados: {champions}',
        f'Margen promedio: {margen_promedio:.1f}%'
    ]

    self._registrar(
        'ANÁLISIS EDA',
        f'{len(self.insights)} insights generados'
    )

    return self


DataPipeline.analizar = analizar

print("Método analizar() registrado")

## Parte 8 — Ejecutar el Pipeline Completo

In [ ]:
print('Iniciando Pipeline DataRetail LATAM...')
pipeline = (DataPipeline('DataRetail_LATAM_M2')
            .ingestar()
            .diagnosticar()
            .limpiar()
            .transformar()
            .analizar())
pipeline.imprimir_log()

## Parte 9 — Dashboard de Resultados

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd

# ==============================
# Obtener datos
# ==============================
df = pipeline.df_trans.copy()
rfm = pipeline.rfm.copy()

# ==============================
# Verificar columnas necesarias
# ==============================
if 'anio' not in df.columns or 'mes' not in df.columns:
    if 'fecha' in df.columns:
        df['fecha'] = pd.to_datetime(df['fecha'])
        df['anio'] = df['fecha'].dt.year
        df['mes'] = df['fecha'].dt.month
    else:
        raise ValueError("No existen las columnas 'anio' y 'mes'.")

if 'trimestre' not in df.columns:
    df['trimestre'] = ((df['mes'] - 1) // 3) + 1

if 'margen_pct' not in df.columns:
    if 'margen_utilidad' in df.columns and 'total_venta' in df.columns:
        df['margen_pct'] = (df['margen_utilidad'] / df['total_venta']) * 100
    else:
        raise ValueError("No existe la información necesaria para calcular margen_pct.")

# ==============================
# Dashboard
# ==============================
fig = plt.figure(figsize=(18,14))
gs = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)

fig.suptitle(
    "Dashboard Final — Proyecto Integrador M2 DataRetail LATAM\nUNICOLOMBO 2026",
    fontsize=14,
    fontweight='bold'
)

# ==============================
# 1. Ventas mensuales
# ==============================
ax1 = fig.add_subplot(gs[0,:])

vm = df.groupby(['anio','mes'],as_index=False)['total_venta'].sum()

vm['anio'] = vm['anio'].astype(int)
vm['mes'] = vm['mes'].astype(int)

vm['fecha'] = pd.to_datetime({
    'year': vm['anio'],
    'month': vm['mes'],
    'day': 1
})

vm = vm.sort_values('fecha')

vm['ma3'] = vm['total_venta'].rolling(window=3,min_periods=1).mean()

ax1.fill_between(vm['fecha'],vm['total_venta'],alpha=0.25)
ax1.plot(vm['fecha'],vm['total_venta'],linewidth=2,label='Ventas')
ax1.plot(vm['fecha'],vm['ma3'],linestyle='--',linewidth=2,label='MA-3')

ax1.set_title("Tendencia de Ventas Mensual",fontweight='bold')
ax1.legend()
plt.setp(ax1.get_xticklabels(),rotation=45)

# ==============================
# 2. Categoría
# ==============================
ax2 = fig.add_subplot(gs[1,0])

df.groupby('categoria')['total_venta'].sum().sort_values().plot(
    kind='barh',
    ax=ax2
)

ax2.set_title("Ventas por Categoría")

# ==============================
# 3. Canal
# ==============================
ax3 = fig.add_subplot(gs[1,1])

df.groupby('canal_venta')['total_venta'].sum().sort_values().plot(
    kind='barh',
    ax=ax3
)

ax3.set_title("Ventas por Canal")

# ==============================
# 4. Segmentos RFM
# ==============================
ax4 = fig.add_subplot(gs[1,2])

pal = {
    'Champions':'green',
    'Loyal':'blue',
    'Potential':'orange',
    'At Risk':'red'
}

rfm['segmento'].value_counts().plot(
    kind='bar',
    ax=ax4,
    color=[pal.get(x,'gray') for x in rfm['segmento'].value_counts().index]
)

ax4.set_title("Segmentos RFM")
ax4.tick_params(axis='x',rotation=45)

# ==============================
# 5. Scatter RFM
# ==============================
ax5 = fig.add_subplot(gs[2,0])

sns.scatterplot(
    data=rfm,
    x='frecuencia',
    y='monetario',
    hue='segmento',
    palette=pal,
    ax=ax5,
    s=40
)

ax5.set_title("Frecuencia vs Monetario")

# ==============================
# 6. Margen
# ==============================
ax6 = fig.add_subplot(gs[2,1])

df['margen_pct'].hist(ax=ax6,bins=30)

ax6.axvline(
    df['margen_pct'].mean(),
    color='red',
    linestyle='--',
    label='Media'
)

ax6.legend()
ax6.set_title("Distribución del Margen %")

# ==============================
# 7. Boxplot
# ==============================
ax7 = fig.add_subplot(gs[2,2])

df.boxplot(
    column='total_venta',
    by='trimestre',
    ax=ax7,
    medianprops=dict(color='red')
)

ax7.set_title("Total Venta por Trimestre")
ax7.set_xlabel("Trimestre")

plt.suptitle("")
plt.tight_layout()

plt.savefig("dashboard_final_m2.png",dpi=120,bbox_inches="tight")
plt.show()

print("Dashboard guardado correctamente.")

## Parte 10 — Reporte Ejecutivo Automatizado

In [ ]:
print('='*70)
print('  REPORTE EJECUTIVO FINAL — DataRetail LATAM | UNICOLOMBO 2026')
print('='*70)
print(f'  Dataset procesado:    {pipeline.metricas["n_raw"]:,} registros brutos')
print(f'  Dataset final:        {pipeline.metricas["n_clean"]:,} registros limpios')
print(f'  Score calidad:        {pipeline.metricas["score_inicial"]}/100 → {pipeline.metricas["score_post"]}/100')
print(f'  Duplicados removidos: {pipeline.metricas["duplicados"]:,}')
print(f'  Nulos imputados:      {pipeline.metricas["nulos"]:,}')
print(f'  Outliers corregidos:  {pipeline.metricas["outliers"]:,}')
print()
print('  INSIGHTS PRINCIPALES:')
for i,insight in enumerate(pipeline.insights,1):
    print(f'  {i}. {insight}')
print('='*70)

## Parte 11 — Exportar Resultados Finales

In [ ]:
pipeline.df_trans.to_csv('DataRetail_LATAM_Final_M2.csv',index=False,encoding='utf-8-sig')
pipeline.rfm.to_csv('RFM_Clientes_Final_M2.csv',index=False,encoding='utf-8-sig')
print('Archivos exportados:')
print(f'  DataRetail_LATAM_Final_M2.csv — {len(pipeline.df_trans):,} filas x {pipeline.df_trans.shape[1]} columnas')
print(f'  RFM_Clientes_Final_M2.csv     — {len(pipeline.rfm):,} clientes segmentados')

## Parte 12 — Reflexiones y Conclusiones del Proyecto

### ¿Qué aprendieron?


Durante el desarrollo de este proyecto aprendimos a construir un pipeline completo de análisis de datos, integrando las etapas de ingesta, diagnóstico, limpieza, transformación y análisis exploratorio (EDA). También fortalecimos nuestras habilidades en el uso de Python y bibliotecas como Pandas, NumPy, Matplotlib y Scikit-learn para preparar datos, generar nuevas variables, segmentar clientes mediante la metodología RFM y construir dashboards que apoyan la toma de decisiones empresariales. Además, comprendimos la importancia de la calidad de los datos para obtener resultados confiables y útiles para el negocio.

###**Desafíos encontrados**


Desafío 1: Manejar valores nulos, registros duplicados y valores atípicos sin afectar la calidad de la información.


Desafío 2: Diseñar una clase DataPipeline reutilizable que integrara todas las etapas del procesamiento de datos.


Desafío 3: Corregir errores en la generación de gráficos y asegurar que todo el notebook ejecutara correctamente desde el inicio hasta el final.
Recomendaciones para DataRetail LATAM


###Recomendación técnica:
 Implementar procesos automáticos de validación y limpieza de datos antes de almacenarlos, reduciendo errores como registros duplicados, datos faltantes e inconsistencias.


###Recomendación de proceso:
Estandarizar la captura de información en todas las sucursales y canales de venta para evitar diferencias en nombres de ciudades, categorías y canales comerciales.


###Recomendación estratégica:
 Fortalecer las estrategias de fidelización dirigidas a los clientes clasificados como Champions y Loyal, además de diseñar campañas específicas para recuperar los clientes At Risk y aumentar las ventas en los meses de menor desempeño.

## Referencias

1. McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly Media.
2. Tukey, J.W. (1977). *Exploratory Data Analysis*. Addison-Wesley.
3. Pedregosa, F. et al. (2011). Scikit-learn. *JMLR*, 12, 2825-2830.
4. Hughes, A.M. (2005). *Strategic Database Marketing*. McGraw-Hill.
5. DAMA International (2017). *DAMA-DMBOK* (2.ª ed.).
6. García, S. et al. (2015). *Data Preprocessing in Data Mining*. Springer.

---
*Proyecto Integrador Final M2 — UNICOLOMBO · Heyder Medrano Olier · 2026*